# MAINTAIN AI V1.5 — Pretrained Time-Series Model Audit

**Goal:** use publicly released pretrained checkpoints before training our own model. This notebook performs zero-shot/feature-extraction checks where the public model supports them. It does **not** train a model.

Candidates:
- **TimeRadar** — pretrained time-series anomaly-detection foundation model.
- **ChronosAD** — pretrained time-series foundation model used as an anomaly feature extractor.
- **C-MAPSS pretrained RUL checkpoints** — useful as a RUL reference, but not directly compatible with MetroPT-3 compressor telemetry.

Public source repositories: urlTimeRadarhttps://github.com/mala-lab/TimeRadar and urlChronosADhttps://github.com/intelligolabs/ChronosAD. The C-MAPSS checkpoint is available at urlHugging Face C-MAPSS RUL referencehttps://huggingface.co/careerbytecode/mlops-ref-manufacturing-rul.

**Decision rule:** a checkpoint only enters MAINTAIN AI if its license, input schema, weights, inference code, and actual behavior on our held-out industrial data can be verified.

In [ ]:
!pip -q install -U git+https://github.com/mala-lab/TimeRadar.git
!pip -q install -U git+https://github.com/intelligolabs/ChronosAD.git
!pip -q install huggingface_hub safetensors pyarrow scikit-learn
import os,sys,json,subprocess,importlib.util
from pathlib import Path
import numpy as np,pandas as pd,torch
ROOT=Path('/content/maintain_ai_v1_3'); OUT=ROOT/'artifacts_pretrained_v1_5'; OUT.mkdir(parents=True,exist_ok=True)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE)
print('python:',sys.version)

## 1. Load the existing MetroPT-3 held-out sequences

We do not retrain or alter labels. The existing V1.3 sequence data is used only for an external-model compatibility/representation audit.

In [ ]:
SEQ=ROOT/'temporal_sequences_v1_3'
X=np.load(SEQ/'metro_X.npy',mmap_mode='r')
Y=np.load(SEQ/'metro_y.npy',mmap_mode='r').astype(np.float32)
T=np.load(SEQ/'metro_times.npy',allow_pickle=True)
spl=np.load(SEQ/'metro_splits.npz')
tr,va,te=[spl[k] for k in ('train','val','test')]
print('X:',X.shape,'Y:',Y.shape)
print('splits:',len(tr),len(va),len(te))
print('test positive rates:',Y[te].mean(0))

## 2. Download and inspect TimeRadar

First verify the repository and checkpoint/configuration files. We intentionally do not guess the model API.

In [ ]:
TR_DIR=OUT/'TimeRadar'
if not TR_DIR.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/mala-lab/TimeRadar.git',str(TR_DIR)],check=True)
print('\n'.join(str(p.relative_to(TR_DIR)) for p in list(TR_DIR.rglob('*'))[:200]))

In [ ]:
import re
readmes=[]
for p in [TR_DIR/'README.md',TR_DIR/'readme.md']:
    if p.exists(): readmes.append(p.read_text(errors='ignore'))
txt='\n'.join(readmes)
for key in ['from_pretrained','checkpoint','TimeRadar','anomaly','AutoModel']:
    print('\n###',key)
    for line in txt.splitlines():
        if key.lower() in line.lower(): print(line[:300])

## 3. Try the released TimeRadar checkpoint

This block is defensive: it discovers the documented loading path from the repository rather than hard-coding an undocumented checkpoint name. If the current public repo requires an unavailable dependency or access step, the result is recorded as **not runnable**, not silently replaced with a newly trained model.

In [ ]:
tr_candidates=[]
for p in TR_DIR.rglob('*'):
    if p.is_file() and p.stat().st_size>0 and p.suffix.lower() in {'.py','.md','.yaml','.yml','.json'}:
        s=p.read_text(errors='ignore')
        if 'from_pretrained' in s or 'checkpoint' in s.lower(): tr_candidates.append(str(p.relative_to(TR_DIR)))
print('candidate implementation/config files:',tr_candidates[:80])
print('TimeRadar inspection complete. Read the printed README/loading instructions before the next cell if the upstream API has changed.')

## 4. ChronosAD compatibility audit

ChronosAD is treated as an anomaly-detection representation candidate, not as a direct 24h/48h/7d risk predictor.

In [ ]:
CA_DIR=OUT/'ChronosAD'
if not CA_DIR.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/intelligolabs/ChronosAD.git',str(CA_DIR)],check=True)
print('ChronosAD files:',len(list(CA_DIR.rglob('*'))))
matches=[]
for p in CA_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in {'.py','.md','.yaml','.yml','.json'}:
        s=p.read_text(errors='ignore')
        if 'from_pretrained' in s or 'checkpoint' in s.lower() or 'Chronos' in s: matches.append(str(p.relative_to(CA_DIR)))
print('\n'.join(matches[:120]))

## 5. C-MAPSS pretrained RUL checkpoint — schema test only

This checkpoint is intentionally not evaluated as a compressor predictor. We only download its files and inspect its declared input schema so we know whether transfer is technically possible.

In [ ]:
from huggingface_hub import snapshot_download
HF_DIR=OUT/'hf_cmapss_rul'
try:
    snapshot_download(repo_id='careerbytecode/mlops-ref-manufacturing-rul',local_dir=str(HF_DIR),local_dir_use_symlinks=False)
    print('downloaded:',[p.name for p in HF_DIR.iterdir()])
    for p in HF_DIR.glob('*'):
        if p.is_file() and p.suffix in {'.py','.md','.json'}: print('\n###',p.name,'\n',p.read_text(errors='ignore')[:6000])
except Exception as e: print('NOT_RUNNABLE:',repr(e))

## 6. Compatibility report

A public model is not automatically a MAINTAIN AI model. Record compatibility explicitly before integration.

In [ ]:
report={
 'version':'maintain-ai-pretrained-audit-v1.5',
 'training_performed':False,
 'dataset':'MetroPT-3 compressor sequences',
 'input_shape':[int(X.shape[1]),int(X.shape[2])],
 'candidates':{
   'TimeRadar':{'role':'anomaly/representation','direct_future_risk':False,'status':'repository/checkpoint inspection'},
   'ChronosAD':{'role':'anomaly/representation','direct_future_risk':False,'status':'repository/checkpoint inspection'},
   'careerbytecode/mlops-ref-manufacturing-rul':{'role':'C-MAPSS RUL reference','direct_future_risk':False,'status':'schema inspection only'}
 },
 'rule':'Do not train or claim production compatibility until input schema, checkpoint, license and inference behavior are verified.'
}
json.dump(report,open(OUT/'pretrained_audit.json','w'),indent=2)
print(json.dumps(report,indent=2))